## Timed responses local LLM

In [ ]:
# Fetch the list of Ollama models from the local Ollama API and print them in a format suitable for use in the OLLAMA_MODELS list.
import requests

def get_ollama_models():
    try:
        response = requests.get("http://localhost:11434/api/tags")
        if response.status_code == 200:
            models = response.json().get("models", [])
            return [model["name"] for model in models]
        else:
            print("Failed to fetch Ollama models.")
            return []
    except Exception as e:
        print(f"Error fetching Ollama models: {e}")
        return []

OLLAMA_MODELS = get_ollama_models()
print(f"You have {len(OLLAMA_MODELS)} models on your computer.\n")
print(f"OLLAMA_MODELS = [")
print('\n'.join(["  " + model + ',' for model in OLLAMA_MODELS]))
print("]")

In [ ]:
# list only the ones you wish to use.
OLLAMA_MODELS = [
    'llama3.2:1b', 
    'llama3.2:latest', 
    'llama3.2:3b',
    'deepseek-r1:1.5b', 
    'smollm:1.7b', 
    'falcon3:3b', 
    'gemma3:1b', 
    'qwen2.5:3b', 
    'phi3:3.8b'
]

In [ ]:
# this is what is needed to use litellm with Ollama.
import litellm
from dotenv import load_dotenv
load_dotenv(override=True)

def message_litellm(model, messages, **kwargs):
    return litellm.completion(
        model=f"ollama/{model}",
        messages=messages,
        **kwargs
    )

In [ ]:
# Function to get timed responses
from datetime import datetime

def get_timed_response(model, messages, **kwargs):
    try:
        start = datetime.now()
        response = message_litellm(model, messages, **kwargs)
        result = response.choices[0].message.content
        end = datetime.now()
        elapsed = (end - start).total_seconds()
        minutes = int(elapsed // 60)
        seconds = elapsed % 60
        heading = f"### {model}  -  {f'{minutes}min ' if minutes > 0 else ''}{seconds:.0f}s\n"
    except Exception as e:
        print(f"Model {model} failed: {e}")
        heading = f"### {model} error: {e}\n"
        return heading, None
    return heading, result

In [9]:
from IPython.display import Markdown, display

prompt = ("Hi!")
messages = [{"role": "user", "content": prompt}]

results = []
for model in OLLAMA_MODELS:
    results.append(get_timed_response(model, messages, temperature=0.2, max_tokens=2000))
    display(Markdown(f"{results[-1][0]}{results[-1][1]}\n\n---\n"))


## Some extra info
`**kwargs` is a Python syntax for handling a variable number of keyword arguments in a function. It collects any extra keyword arguments passed to the function into a dictionary called `kwargs`. In the code, it's used to pass additional parameters (like `temperature` and `max_tokens`) to the `litellm.completion` call without explicitly defining them in the function signature. For example:

- In `get_response`, `**kwargs` allows passing extra options to `litellm.completion`.
- In `artist`, `**kwargs` forwards those options to `get_response`.`litellm.completion` supports a wide range of inputs, unifying parameters across providers like OpenAI, Anthropic, Ollama, etc. Beyond `model`, `messages`, and `stream` (as seen in your code), common supported parameters include:

- `temperature` (float, e.g., 0.2): Controls randomness in responses.
- `max_tokens` (int, e.g., 2000): Limits the response length.
- `top_p` (float): Nucleus sampling parameter.
- `frequency_penalty` (float): Reduces repetition of frequent tokens.
- `presence_penalty` (float): Reduces repetition of any tokens.
- `stop` (list of strings): Sequences to stop generation at.
- `logprobs` (bool or int): Returns log probabilities.
- `echo` (bool): Includes the prompt in the response.
- `n` (int): Number of completions to generate.
- `best_of` (int): For sampling multiple and returning the best.
- `logit_bias` (dict): Biases token probabilities.
- `user` (string): User identifier for tracking.

For Ollama-specific models, additional options like `num_ctx` (context window size) or `num_predict` may be supported via `**kwargs`. Check the [LiteLLM documentation](https://docs.litellm.ai/) for the full list and provider-specific details.